In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client  = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

Usually yes — if the course is still open or has not passed its enrollment deadline, you can likely join it.

If you want, I can help you figure out the exact answer. Just tell me:
- the course name
- the platform or school
- whether it’s live, online, or in-person
- when it starts or started

Then I can help you check if late enrollment is possible.


In [4]:
def search(query):
    boost_dict  = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered enrollment can I join now"}', call_id='call_QPJ2e9VJhOZk7XU1YEkqb3XY', name='search', type='function_call', id='fc_0ea4137954a99a60006a6753dd999881a388f0ea5cf13da702', caller=None, namespace=None, status='completed')]

In [7]:
import json

In [8]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered enrollment can I join now"}', call_id='call_QPJ2e9VJhOZk7XU1YEkqb3XY', name='search', type='function_call', id='fc_0ea4137954a99a60006a6753dd999881a388f0ea5cf13da702', caller=None, namespace=None, status='completed')

In [9]:
args = json.loads(call.arguments)
args

{'query': 'join course discovered enrollment can I join now'}

In [10]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [11]:
results_json = json.dumps(results, indent=2)
results_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nTo get the certificate, you need to finish a capstone project and complete the\\nrequired peer reviews. Homework is not required. You can work through the\\nmaterial and prepare your project in self-paced mode, but project submission and\\npeer review must happen while a live cohort is accepting them."\n  },\n  {\n    "id": "977bf7786c",\n    "

In [12]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [13]:
messages.extend(response.output)

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered enrollment can I join now"}', call_id='call_QPJ2e9VJhOZk7XU1YEkqb3XY', name='search', type='function_call', id='fc_0ea4137954a99a60006a6753dd999881a388f0ea5cf13da702', caller=None, namespace=None, status='completed')]

In [14]:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json,
})

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered enrollment can I join now"}', call_id='call_QPJ2e9VJhOZk7XU1YEkqb3XY', name='search', type='function_call', id='fc_0ea4137954a99a60006a6753dd999881a388f0ea5cf13da702', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_QPJ2e9VJhOZk7XU1YEkqb3XY',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificat

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes, you can still join and start learning.

If you want a certificate, you’ll need to submit your project while the course is still accepting submissions.


In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(770, 35)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15 / 1_000_000
    OUTPUT_PRICE_PER_MILLION = 0.60 / 1_000_000

    input_cost = (input_tokens) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [18]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [20]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


In [21]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [22]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enroll registration"}
function_call: search {"query":"course enrollment late join discovered course FAQ"}


In [23]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [24]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure you submit your project while the course is still accepting submissions. You can also start learning and working through the materials at any time.

If you’d like, I can also help you with how to start the course or explain the certificate requirements.


In [25]:
iteration = 1

while True:
    print(f"Iteration #{iteration}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    iteration += 1
    if has_function_calls == False:
        break

    

Iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure to submit your project while submissions are still being accepted. You can also start learning and working through the materials anytime.

If you want, I can also help with how to start the course or explain the certificate requirements.


In [26]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    iteration = 1

    while True:
        print(f"Iteration #{iteration}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        iteration = iteration + 1
        if has_function_calls == False:
            break

    return last_answer

In [27]:
agent_loop(instructions, "How do I run Olama locally?")

Iteration #1...
function_call: search {"query":"Olama locally run install local model Ollama"}
Iteration #2...
function_call: search {"query":"ollama serve localhost 11434 chat llama3 local server install shell"}
Iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS:** download the `.pkg` from https://ollama.com/download
   - **Windows:** download the `.msi`
   - **Linux:** run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model and starts a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a JSON response showing available models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(respon

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS:** download the `.pkg` from https://ollama.com/download\n   - **Windows:** download the `.msi`\n   - **Linux:** run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and starts a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a JSON response showing available models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n```\nOr in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 

In [28]:
result = agent_loop(instructions, "I just discovered the course. Can I still join it?")

Iteration #1...
function_call: search {"query":"can I still join the course late enroll discovered the course can I still join"}
Iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want to receive a certificate, though, you’ll need to submit your project while submissions are still being accepted.

If you’d like, I can also help you figure out the best way to start catching up. Are there other areas you want to explore?


In [29]:
print(result)

Yes — you can still join the course.

If you want to receive a certificate, though, you’ll need to submit your project while submissions are still being accepted.

If you’d like, I can also help you figure out the best way to start catching up. Are there other areas you want to explore?


In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()


agent_loop(instructions, "I just discovered the course. Can I join it?")

Iteration #1...
function_call: search {"query":"join course discovered course can I join enrolled registration late join FAQ"}
Iteration #2...
ASSISTANT:
Yes — you can still join the course.

According to the course FAQ, if you just discovered it, you’re welcome to start learning and participate. One important note: if you want a certificate, you need to submit your project while submissions are still being accepted.

If you want, I can also help with what “joining” means in practice, like registration, homework, or certificates. Any other areas you want to explore?


'Yes — you can still join the course.\n\nAccording to the course FAQ, if you just discovered it, you’re welcome to start learning and participate. One important note: if you want a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help with what “joining” means in practice, like registration, homework, or certificates. Any other areas you want to explore?'

In [31]:
agent_loop(instructions, "what's queen gambit?")

Iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is it"}
Iteration #2...
function_call: search {"query":"\"queen's gambit\" chess opening definition"}
Iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to pull Black’s **d-pawn** away from the center and gain more control of the board.

A few quick points:
- It’s called a **gambit** because White is “sacrificing” a pawn, at least temporarily.
- Black can accept it or decline it.
- It’s one of the most famous and classical chess openings.

If you want, I can also explain:
- the main ideas behind it,
- the difference between **Queen’s Gambit Accepted** and **Declined**,
- or show a simple opening line.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to pull Black’s **d-pawn** away from the center and gain more control of the board.\n\nA few quick points:\n- It’s called a **gambit** because White is “sacrificing” a pawn, at least temporarily.\n- Black can accept it or decline it.\n- It’s one of the most famous and classical chess openings.\n\nIf you want, I can also explain:\n- the main ideas behind it,\n- the difference between **Queen’s Gambit Accepted** and **Declined**,\n- or show a simple opening line.'

In [32]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


agent_loop(instructions, "what's queen gambit?")

Iteration #1...
function_call: search {"query":"queen gambit"}
Iteration #2...
function_call: search {"query":"queen's gambit chess opening course FAQ"}
Iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen’s gambit,” so it looks like that’s outside the course/logistics topics I can answer from the FAQ.

If you meant a course-related term, feel free to rephrase it and I’ll check again. Are there other areas you want to explore?


'I couldn’t find anything in the course FAQ about “queen’s gambit,” so it looks like that’s outside the course/logistics topics I can answer from the FAQ.\n\nIf you meant a course-related term, feel free to rephrase it and I’ll check again. Are there other areas you want to explore?'

### Lesson 15

In [33]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [34]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [35]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )

In [36]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [37]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [38]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [39]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [40]:
result = runner.loop(
    prompt = "How do I run Olama locally?",
    callback=callback
)

-> Response received


-> Response received


-> Response received


In [41]:
result

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Olama locally run local Ollama install r

In [42]:
result.cost

CostInfo(input_cost=Decimal('0.00331425'), output_cost=Decimal('0.001422'), total_cost=Decimal('0.00473625'))

In [43]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local Ollama install run locally"}', call_id

In [46]:
result2 = runner.loop(
    prompt = "How do I run a different model?",
    previous_messages = result.all_messages,
    callback = callback
)

-> Response received


-> Response received


In [47]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='how do i get a certeficete', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"certificate how do i get a certificate ce